# 0. Problem
## 1731. The Number of Employees Which Report to Each Employee — Easy
For each manager with direct reports, return manager ID/name, report count, and rounded average report age.

Official: https://leetcode.com/problems/the-number-of-employees-which-report-to-each-employee/

# 1. Setup

In [ ]:
import pandas as pd
employees_rows=[(9,"Hercy",None,43),(6,"Alice",9,41),(4,"Bob",9,36),(2,"Winston",None,37),(7,"David",2,37),(8,"Jonathan",2,39)]
employees_pd=pd.DataFrame(employees_rows,columns=["employee_id","name","reports_to","age"])
employees_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
employees_spark=spark.createDataFrame(employees_rows,"employee_id int, name string, reports_to int, age int")
employees_spark.createOrReplaceTempView("Employees")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT m.employee_id,m.name,COUNT(r.employee_id) AS reports_count,ROUND(AVG(r.age)) AS average_age
FROM Employees m
JOIN Employees r ON m.employee_id=r.reports_to
GROUP BY m.employee_id,m.name
ORDER BY m.employee_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
reports_pd=(employees_pd.dropna(subset=["reports_to"]).groupby("reports_to",as_index=False).agg(reports_count=("employee_id","count"),average_age=("age","mean")).rename(columns={"reports_to":"employee_id"}))
reports_pd["employee_id"]=reports_pd["employee_id"].astype(int)
reports_pd["average_age"]=reports_pd["average_age"].round().astype(int)
result_pd=(reports_pd.merge(employees_pd[["employee_id","name"]],on="employee_id")[["employee_id","name","reports_count","average_age"]].sort_values("employee_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
result_spark=(employees_spark.alias("m").join(employees_spark.alias("r"),F.col("m.employee_id")==F.col("r.reports_to"),"inner").groupBy(F.col("m.employee_id").alias("employee_id"),F.col("m.name").alias("name")).agg(F.count(F.col("r.employee_id")).alias("reports_count"),F.round(F.avg(F.col("r.age"))).cast("int").alias("average_age")).orderBy("employee_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| manager/report relation | self join | group by `reports_to` + merge | self join |
| rounded average | `ROUND(AVG())` | `.mean().round()` | `F.round(F.avg())` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Employees

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: employees_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: employees_spark